# 00 · Foundations — Part 3: Schemas, Validation and Runtime Context
### LangGraph · designing what flows through your graph

Parts 1 and 2 used a `TypedDict` for state without asking whether that was the best choice. This notebook
answers three design questions you face the moment a graph leaves a notebook and becomes a real service:

1. **What kind of object should my state be**, and what does each choice actually *check* for me?
2. **Who gets to see what?** The caller of your graph shouldn't need to send, or receive, your internal
   scratch fields.
3. **Where do settings go** (a customer's name, a tone of voice, a deadline table) when they aren't part of
   the data being processed?

> 📌 **Checked while building this notebook** (`langgraph 1.2.17`; every point below was observed by running it)
> - **Only a Pydantic state rejects wrongly-typed input.** `TypedDict` and dataclass states accepted
>   `n="not a number"` without complaint.
> - **Nodes receive different things:** a `dict` for `TypedDict`, an object for a dataclass or Pydantic
>   model. In every case **`invoke` returns a plain `dict`**.
> - **Validation covers the input only, not what nodes write.** A node that wrote `age=-1` into a Pydantic
>   state with `ge=0` was *not* stopped. And a **misspelled key is silently dropped, even with Pydantic.**
> - **A Pydantic `input_schema` did not validate**; validation happened only when the *state* class itself was
>   a Pydantic model. So: Pydantic state (validates) + `output_schema` (hides private fields).
> - **Forgetting to pass `context=` gives a confusing error**, `'NoneType' object has no attribute ...`.
> - `config_schema` is deprecated since LangGraph 1.0; use `context_schema`.
> - Everything here runs offline with no API key.

## Setup

In [ ]:
# %pip install -U langgraph pydantic


In [ ]:
from dataclasses import dataclass, field
from typing import TypedDict, Literal

from pydantic import BaseModel, Field, ValidationError
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime

print("Ready.")


---
## 1. Three ways to declare state

LangGraph accepts three kinds of state class. Here is the *same tiny graph* (decide a priority from a subject
line) written all three ways, so you can compare them directly.

In [ ]:
seen = {}                      # records what each node actually receives


# --- 1) TypedDict: a dictionary with documented keys -----------------------------------------
class TypedState(TypedDict):
    subject: str
    priority: str

def typed_node(state: TypedState) -> dict:
    seen["TypedDict"] = type(state).__name__
    return {"priority": "high" if "urgent" in state["subject"].lower() else "low"}          # state["key"]


# --- 2) dataclass: attribute access, with defaults ---------------------------------------------
@dataclass
class DataState:
    subject: str
    priority: str = ""

def data_node(state: DataState) -> dict:
    seen["dataclass"] = type(state).__name__
    return {"priority": "high" if "urgent" in state.subject.lower() else "low"}             # state.key


# --- 3) Pydantic model: attribute access, plus validation --------------------------------------
class ModelState(BaseModel):
    subject: str
    priority: str = ""

def model_node(state: ModelState) -> dict:
    seen["Pydantic"] = type(state).__name__
    return {"priority": "high" if "urgent" in state.subject.lower() else "low"}             # state.key


def make(schema, node):
    b = StateGraph(schema)
    b.add_node("decide", node)
    b.add_edge(START, "decide"); b.add_edge("decide", END)
    return b.compile()


for label, schema, node in [("TypedDict", TypedState, typed_node), ("dataclass", DataState, data_node), ("Pydantic", ModelState, model_node)]:
    result = make(schema, node).invoke({"subject": "URGENT: site down"})
    print(f"{label:10} node received a {seen[label]:11} invoke returned a {type(result).__name__}: {result}")


Same answer, three different shapes. A quick comparison (all measured):

| | Node receives | `invoke` returns | Rejects a wrong-typed input? | Best for |
|---|---|---|---|---|
| **`TypedDict`** | a `dict` | a `dict` | No | Simple graphs; the default in most examples |
| **dataclass** | an object (`state.subject`) | a `dict` | No | Attribute access and defaults, without Pydantic |
| **Pydantic** | an object (`state.subject`) | a `dict` | **Yes** | Graphs that take input from outside (an API, a form) |

---
## 2. What "validation" really gives you

With Pydantic state, **input** is checked before any node runs. Let's give it a rule (`age` can't be negative)
and try to break it three ways.

In [ ]:
class Customer(BaseModel):
    name: str
    age: int = Field(ge=0)               # must be 0 or more


def birthday(state: Customer) -> dict:
    return {"age": state.age + 1}

b = StateGraph(Customer); b.add_node("birthday", birthday); b.add_edge(START, "birthday"); b.add_edge("birthday", END)
customer_graph = b.compile()

print("good input     ->", customer_graph.invoke({"name": "Asha", "age": 20}))

try:
    customer_graph.invoke({"name": "Asha", "age": -5})
except ValidationError as error:
    print("bad input      -> rejected:", error.errors()[0]["msg"])


That's the protection you wanted. **But it only covers what comes *in*.** Watch what a node can still do:

In [ ]:
def corrupt(state: Customer) -> dict:
    return {"age": -1}                    # breaks the rule we declared

b = StateGraph(Customer); b.add_node("corrupt", corrupt); b.add_edge(START, "corrupt"); b.add_edge("corrupt", END)
print("node writes -1 ->", b.compile().invoke({"name": "Asha", "age": 20}), " <- no error: node output isn't validated")


def typo(state: Customer) -> dict:
    return {"agee": 99}                   # misspelled key

b = StateGraph(Customer); b.add_node("typo", typo); b.add_edge(START, "typo"); b.add_edge("typo", END)
print("misspelled key ->", b.compile().invoke({"name": "Asha", "age": 20}), " <- silently dropped")


So treat Pydantic state as a **gatekeeper at the front door**, not a guard on every room. To protect what
a node *writes*, build the update through a model **inside the node**, so a bad value raises right there:

In [ ]:
class AgeUpdate(BaseModel):
    age: int = Field(ge=0)


def careful(state: Customer) -> dict:
    return AgeUpdate(age=state.age - 100).model_dump()      # raises if the result would be negative

b = StateGraph(Customer); b.add_node("careful", careful); b.add_edge(START, "careful"); b.add_edge("careful", END)
try:
    b.compile().invoke({"name": "Asha", "age": 20})
except ValidationError as error:
    print("caught inside the node:", error.errors()[0]["msg"])


For the **typo** problem there is no runtime guard, but your editor can catch it. Describe the allowed updates
with a `TypedDict` that has `total=False` (every key optional) and use it as the node's return type. A type
checker such as `mypy` or `pyright` will then flag `{"agee": 99}` before you ever run the code:

```python
class CustomerUpdate(TypedDict, total=False):
    name: str
    age: int

def birthday(state: Customer) -> CustomerUpdate:      # a misspelled key here is a type error
    return {"age": state.age + 1}
```

(This is a static check only. At runtime, the misspelled key would still be dropped silently.)

---
## 3. Public vs. private state

Your graph's *working* state is full of scratch fields: cleaned text, flags, notes. The code that **calls** the
graph shouldn't have to know about them, and you may not want to hand them back either.

`output_schema` limits what `invoke` returns. Combine it with a Pydantic state and you get both things at once:
**validated input, and a hidden interior.**

In [ ]:
class Work(BaseModel):                         # the full working state
    subject: str = Field(min_length=1)         # must not be empty
    body: str
    reply: str = ""
    internal_notes: str = ""                   # private scratch space


class PublicResult(TypedDict):                 # the only fields the caller gets back
    reply: str


def respond(state: Work) -> dict:
    return {"reply": f"Re: {state.subject}", "internal_notes": "matched rule #4, flagged for review"}


b = StateGraph(Work, output_schema=PublicResult)
b.add_node("respond", respond); b.add_edge(START, "respond"); b.add_edge("respond", END)
api = b.compile()

print("returned    :", api.invoke({"subject": "Refund", "body": "please"}))

try:
    api.invoke({"subject": "", "body": "x"})
except ValidationError as error:
    print("empty subject rejected:", error.errors()[0]["msg"])


The caller got only `reply`; `internal_notes` stayed inside. (There is also an `input_schema`, which declares
which fields the caller may send. We found that giving it a Pydantic class did *not* add validation on its own;
the validation came from the state class, which is why the pattern above is built that way.)

---
## 4. Nodes can ask for more than the state

So far each node took one argument, `state`. A node can also ask for **`config`**, which carries per-run
details such as a conversation's `thread_id` (you'll use it for memory in `07_Memory_Persistence`):

In [ ]:
class Echo(TypedDict):
    out: str


def who_am_i(state: Echo, config) -> dict:
    return {"out": f"running on thread {config['configurable'].get('thread_id')}"}

b = StateGraph(Echo); b.add_node("n", who_am_i); b.add_edge(START, "n"); b.add_edge("n", END)
print(b.compile().invoke({"out": ""}, config={"configurable": {"thread_id": "ticket-4471"}}))


---
## 5. Settings that aren't state: runtime context

State is the **data being processed**; it changes as the graph runs. But some things stay fixed for a whole run:
which customer's account this is, what tone to write in, which deadlines apply. Putting those in state mixes
settings with data, and makes them part of every saved snapshot.

LangGraph has a separate channel for them: **runtime context**. You declare its shape with `context_schema`,
pass the values when you run (`context=`), and a node reads them through a `runtime` argument.

| | **State** | **Runtime context** |
|---|---|---|
| Changes during the run? | Yes, nodes update it | No, read-only |
| Typical contents | the ticket, flags, messages | tenant name, tone, user id, model name |
| Declared with | `StateGraph(State)` | `StateGraph(State, context_schema=Ctx)` |
| Supplied at | `invoke({...})` | `invoke({...}, context=Ctx(...))` |
| Read in a node | `state` | `runtime.context` |

In [ ]:
@dataclass
class Tone:
    tenant: str = "default"
    style: Literal["formal", "friendly"] = "friendly"


class Greeting(TypedDict):
    reply: str


def greet(state: Greeting, runtime: Runtime[Tone]) -> dict:
    ctx = runtime.context
    opening = "Dear customer," if ctx.style == "formal" else "Hi there!"
    return {"reply": f"{opening} (from {ctx.tenant})"}


b = StateGraph(Greeting, context_schema=Tone)
b.add_node("greet", greet); b.add_edge(START, "greet"); b.add_edge("greet", END)
greeter = b.compile()

print(greeter.invoke({"reply": ""}, context=Tone(tenant="ShopEasy", style="friendly")))
print(greeter.invoke({"reply": ""}, context=Tone(tenant="BankCo",  style="formal")))


One graph, two behaviours, and **nothing about the tenant touched the state.**

### The gotcha: forgetting the context

If you call the graph without `context=`, `runtime.context` is `None`, and you get a confusing error far from the
real mistake:

In [ ]:
try:
    greeter.invoke({"reply": ""})
except Exception as error:
    print(type(error).__name__, "->", error)


A defensive habit: fall back to a default inside the node, so a forgotten `context=` degrades gracefully:

```python
ctx = runtime.context or Tone()          # Tone() has defaults for every field
```

(Older tutorials pass settings through `config_schema` and `config["configurable"]`. That still works but is
deprecated since LangGraph 1.0 in favour of `context_schema`.)

---
## Real-world capstone: a "Ticket API" with validated input, private internals and per-customer settings

A support platform serves several companies. Each has its own **tone of voice** and its own **response deadlines**.
We'll build one graph that:

- **rejects malformed tickets** at the door (Pydantic state),
- **keeps its working notes private** (`output_schema`),
- **behaves differently per company** without any of that leaking into state (`context_schema`).

```
 START → analyse → draft_reply → set_deadline → END
```

In [ ]:
@dataclass
class Tenant:
    """Per-company settings: read-only for the whole run."""
    name: str = "Acme"
    tone: Literal["formal", "friendly"] = "friendly"
    sla_hours: dict = field(default_factory=lambda: {"high": 4, "medium": 24, "low": 72})


class TicketWork(BaseModel):
    """The graph's working state. Pydantic, so incoming tickets are validated."""
    subject: str = Field(min_length=1)
    body: str = Field(min_length=1)
    clean_text: str = ""
    queue: str = ""
    priority: str = ""
    reply: str = ""
    sla_hours: int = 0
    internal_notes: str = ""             # never returned to the caller


class TicketResult(TypedDict):
    """The public answer."""
    queue: str
    priority: str
    reply: str
    sla_hours: int


def analyse(state: TicketWork) -> dict:
    text = " ".join(f"{state.subject} {state.body}".split()).lower()
    if any(w in text for w in ("charge", "refund", "invoice", "payment")):
        queue = "billing"
    elif any(w in text for w in ("crash", "error", "bug", "not working")):
        queue = "technical"
    else:
        queue = "general"
    if any(w in text for w in ("urgent", "twice", "fraud", "not working")):
        priority = "high"
    elif queue != "general":
        priority = "medium"
    else:
        priority = "low"
    return {"clean_text": text, "queue": queue, "priority": priority,
            "internal_notes": f"rule match -> queue={queue}, priority={priority}"}


def draft_reply(state: TicketWork, runtime: Runtime[Tenant]) -> dict:
    tenant = runtime.context or Tenant()
    hours = tenant.sla_hours[state.priority]
    unit = "hour" if hours == 1 else "hours"
    if tenant.tone == "formal":
        reply = (f"Dear customer, thank you for contacting {tenant.name}. Your {state.queue} request has been "
                 f"logged and will be answered within {hours} {unit}.")
    else:
        reply = (f"Hi there! Thanks for reaching out to {tenant.name}. We've sent your {state.queue} request to "
                 f"our team and you'll hear back within {hours} {unit}.")
    return {"reply": reply}


def set_deadline(state: TicketWork, runtime: Runtime[Tenant]) -> dict:
    tenant = runtime.context or Tenant()
    return {"sla_hours": tenant.sla_hours[state.priority]}


builder = StateGraph(TicketWork, output_schema=TicketResult, context_schema=Tenant)
builder.add_node("analyse", analyse)
builder.add_node("draft_reply", draft_reply)
builder.add_node("set_deadline", set_deadline)
builder.add_edge(START, "analyse")
builder.add_edge("analyse", "draft_reply")
builder.add_edge("draft_reply", "set_deadline")
builder.add_edge("set_deadline", END)

ticket_api = builder.compile()
print(ticket_api.get_graph().draw_mermaid())


In [ ]:
ticket = {"subject": "Charged twice", "body": "I was charged twice for my order, please refund one."}

shopeasy = Tenant(name="ShopEasy", tone="friendly")
bankco = Tenant(name="BankCo", tone="formal", sla_hours={"high": 1, "medium": 8, "low": 24})

result_a = ticket_api.invoke(ticket, context=shopeasy)
result_b = ticket_api.invoke(ticket, context=bankco)

print("ShopEasy:", result_a, "\n")
print("BankCo  :", result_b)


In [ ]:
# 1. The same ticket produced different replies and deadlines, purely because of the context
assert result_a["queue"] == result_b["queue"] == "billing"
assert result_a["priority"] == result_b["priority"] == "high"
assert (result_a["sla_hours"], result_b["sla_hours"]) == (4, 1)
assert "Hi there" in result_a["reply"] and "Dear customer" in result_b["reply"]

# 2. Private working fields never leave the graph
assert set(result_a) == {"queue", "priority", "reply", "sla_hours"}
assert "internal_notes" not in result_a and "clean_text" not in result_a

# 3. Malformed tickets are rejected before any node runs
try:
    ticket_api.invoke({"subject": "", "body": "no subject"}, context=shopeasy)
    raise AssertionError("an empty subject should have been rejected")
except ValidationError as error:
    print("rejected:", error.errors()[0]["msg"], "(field:", error.errors()[0]["loc"][0] + ")")

# 4. Forgetting the context degrades gracefully, because the nodes fall back to Tenant()
print("no context ->", ticket_api.invoke(ticket)["reply"])
print("\nAll checks passed.")


Three design decisions, each doing one job:

- **Pydantic state** is the *front door*: bad input never reaches a node.
- **`output_schema`** is the *window*: callers see the answer and none of the scratch work.
- **`context_schema`** is the *settings panel*: per-customer behaviour without cluttering the data.

Notice what didn't change when we served a second company: the graph, the nodes, the state. Only the `context`
passed to `invoke`.

### Try it yourself
1. Add a third tenant with `tone="friendly"` but `sla_hours={"high": 2, "medium": 12, "low": 48}`. Which
   fields of the result change?
2. Add a field `language: Literal["en", "hi"] = "en"` to `Tenant` and make `draft_reply` reply in Hindi when
   it's `"hi"`.
3. Make `analyse` return `{"priorty": "high"}` (a typo). What does the final result look like, and why?
   Then add a `total=False` update type and see whether your editor flags it.
4. Remove `output_schema=TicketResult` from `StateGraph(...)`. What does `invoke` return now?
5. Change `TicketWork` from Pydantic to a `TypedDict` and invoke with an empty subject. What happens to the
   door?

---
**You've finished Foundations.** You can now describe a workflow as state, nodes and edges; run branches in
parallel and join them safely; combine writes with reducers; manage a conversation's messages; and choose,
validate and expose your state deliberately.

**Next:** `01_Sequential_Workflows/` — the first of the workflow patterns, now with all of this vocabulary
in place.
